# Complete Momentum Strategy Guide

This notebook provides an end-to-end exploration of **momentum strategies** in futures markets.

## What You'll Learn

1. **Momentum Anomaly** - Why momentum works in financial markets
2. **Multi-Period Momentum** - Testing different lookback windows (5, 10, 20, 60 days)
3. **Time-Series vs Cross-Sectional** - Two approaches to momentum
4. **Signal Combination** - Combining multiple momentum signals
5. **Volatility Adjustment** - Scaling positions by risk
6. **Performance Analysis** - Comparing momentum to carry strategies
7. **Parameter Sensitivity** - Finding optimal lookback periods
8. **Turnover Analysis** - Understanding trading frequency and costs

## Prerequisites

Complete `01_getting_started.ipynb` first to understand the framework basics.

---

## Part 1: Understanding the Momentum Anomaly

### What is Momentum?

**Momentum** is the tendency for assets that have performed well (poorly) in the recent past to continue performing well (poorly) in the near future.

### Why Does Momentum Work?

Several behavioral and structural factors explain momentum:

1. **Behavioral Biases**:
   - **Under-reaction**: Investors initially under-react to news, causing prices to drift
   - **Herding**: Trend-following behavior amplifies price movements
   - **Anchoring**: Investors anchor to past prices and adjust slowly

2. **Risk Premia**:
   - Momentum may be compensation for systematic risk exposure
   - "Bad times" when trend-following loses money (rapid reversals)

3. **Market Microstructure**:
   - Slow information diffusion across markets
   - Time-varying risk premia

### Historical Evidence

**Moskowitz, Ooi & Pedersen (2012)** - "Time Series Momentum":
- Momentum works across 58 liquid futures markets
- 1-12 month lookback optimal for most assets
- Sharpe ratio ~0.8 after transaction costs
- Low correlation with equity market risk

**Jegadeesh & Titman (1993)** - Cross-sectional momentum in stocks:
- 3-12 month formation periods work best
- Partially reverses after 12 months

### Time-Series vs Cross-Sectional Momentum

**Time-Series Momentum (Trend Following)**:
- Compare asset to its own past
- Signal: Is this asset going up or down?
- Works in trending markets
- Example: If SFRZ4 up 2% last month → buy SFRZ4

**Cross-Sectional Momentum (Relative Strength)**:
- Compare assets to each other
- Signal: Which asset is strongest?
- Works in mean-reverting market regimes
- Example: If SFRZ4 outperformed peers → buy SFRZ4

### Expected Performance Characteristics

- **Information Coefficient (IC)**: 0.02 - 0.05 (moderate skill)
- **Sharpe Ratio**: 0.5 - 1.0 (after costs)
- **Turnover**: High (50-100% monthly)
- **Risk**: Fat tails, trend reversals, momentum crashes

---

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
import numpy as np
import pandas as pd
from datetime import date, timedelta
from typing import Dict, List

# Signal imports
from Signals.Futures.MomentumSignal import MomentumSignal
from Signals.Futures.CarrySignal import CarrySignal
from Signals.SignalCombiner import SignalCombiner

# Adapter imports
from Adapter.FuturesAdapter import FuturesAdapter

# Strategy imports
from Strategies.Registry import quick_strategy
from Backtest.Backtest import Backtest

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
np.random.seed(42)

print("✓ Setup complete!")

---

## Part 2: Create Enhanced Mock Market Data

We'll create synthetic market data with realistic momentum characteristics:
- Trending behavior (autocorrelation)
- Carry structure
- Realistic volatility

In [ ]:
class MomentumMockMDP:
    """
    Mock market data provider with momentum characteristics.
    
    Features:
    - Price history with trends
    - Carry structure (calendar spreads)
    - Autocorrelated returns (momentum)
    """
    
    def __init__(self, base_rate=5.0, carry_spread=0.10, momentum_strength=0.3, seed=42):
        """
        Initialize mock market data.
        
        Args:
            base_rate: Base interest rate level
            carry_spread: Calendar spread (bps per quarter)
            momentum_strength: Autocorrelation coefficient (0-1)
            seed: Random seed for reproducibility
        """
        self.base_rate = base_rate
        self.carry_spread = carry_spread
        self.momentum_strength = momentum_strength
        np.random.seed(seed)
        
        # Store price history for each contract
        self.price_history = {}
        self.last_returns = {}  # For autocorrelation
    
    def get_pricer(self, currency, as_of):
        return self
    
    def futures_price(self, contract, as_of=None):
        """
        Generate futures price with momentum.
        
        Price evolution:
        1. Base price from carry structure
        2. Add momentum (autocorrelated returns)
        3. Add random noise
        """
        # Extract contract info
        quarter_map = {'H': 0, 'M': 1, 'U': 2, 'Z': 3}
        quarter_code = contract[-2] if len(contract) >= 2 else 'H'
        quarter = quarter_map.get(quarter_code, 0)
        
        # Base price from carry structure
        rate = self.base_rate + (quarter * self.carry_spread)
        base_price = 100.0 - rate
        
        # Initialize price history if needed
        if contract not in self.price_history:
            self.price_history[contract] = [base_price]
            self.last_returns[contract] = 0.0
        
        # Generate return with momentum (autocorrelation)
        last_return = self.last_returns[contract]
        innovation = np.random.normal(0, 0.02)  # Random shock
        new_return = self.momentum_strength * last_return + innovation
        
        # Update price
        last_price = self.price_history[contract][-1]
        new_price = last_price * (1 + new_return)
        
        # Store
        self.price_history[contract].append(new_price)
        self.last_returns[contract] = new_return
        
        return new_price
    
    def get_price_history(self, instrument, start_date, end_date):
        """
        Get historical prices for an instrument.
        
        Returns:
            Polars DataFrame with 'date' and 'price' columns
        """
        import polars as pl
        
        # Ensure we have enough history
        if instrument not in self.price_history:
            # Generate initial history
            for _ in range(100):
                self.futures_price(instrument)
        
        # Get prices (use last N points)
        days = (end_date - start_date).days + 10
        prices = self.price_history[instrument][-days:]
        
        # Create date range
        dates = pd.date_range(end=end_date, periods=len(prices), freq='D')
        
        # Return as Polars DataFrame
        return pl.DataFrame({
            'date': dates,
            'price': prices
        })

# Create market data with momentum
mdp = MomentumMockMDP(
    base_rate=5.0, 
    carry_spread=0.10,
    momentum_strength=0.3,  # 30% autocorrelation
    seed=42
)

# Define universe
instruments = ['SFRZ4', 'SFRH5', 'SFRM5', 'SFRU5', 'SFRZ5']

# Generate some initial price history
for inst in instruments:
    for _ in range(200):
        mdp.futures_price(inst)

print("✓ Mock market data created")
print(f"  Universe: {len(instruments)} contracts")
print(f"  Momentum strength: {mdp.momentum_strength} (autocorrelation)")
print(f"  Price history: {len(mdp.price_history['SFRZ4'])} periods per contract")

### Visualize Price Trends

Let's examine the generated price series to confirm momentum characteristics:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Price levels
ax1 = axes[0]
for inst in instruments:
    prices = mdp.price_history[inst][-100:]  # Last 100 periods
    ax1.plot(prices, label=inst, linewidth=2, alpha=0.7)
ax1.set_title('Futures Price Evolution (Last 100 Periods)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price', fontsize=12)
ax1.set_xlabel('Period', fontsize=12)
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Returns (showing autocorrelation)
ax2 = axes[1]
for inst in instruments[:2]:  # Just show 2 for clarity
    prices = np.array(mdp.price_history[inst][-100:])
    returns = np.diff(prices) / prices[:-1]
    ax2.plot(returns, label=inst, linewidth=1.5, alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax2.set_title('Returns (Showing Momentum/Autocorrelation)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Return', fontsize=12)
ax2.set_xlabel('Period', fontsize=12)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("  • Notice trending behavior in prices (not pure random walk)")
print("  • Returns show persistence (positive follows positive)")
print("  • This is the momentum effect we want to capture!")

---

## Part 3: Multi-Period Momentum Signals

We'll test momentum with different lookback periods:
- **5 days** - Very short-term (microstructure, mean reversion risk)
- **10 days** - Short-term (2 weeks, common in futures)
- **20 days** - Medium-term (1 month, classic momentum)
- **60 days** - Long-term (3 months, Moskowitz et al. 2012)

In [ ]:
# Create momentum signals with different lookbacks
lookback_periods = [5, 10, 20, 60]
momentum_signals = {}

for lookback in lookback_periods:
    signal = MomentumSignal(
        name=f"momentum_{lookback}d",
        lookback_days=lookback,
        method='simple',  # Simple returns
        annualize=False,  # Keep as raw returns for now
        standardize=True  # Z-score normalization
    )
    momentum_signals[lookback] = signal

print("Momentum Signals Created:")
print("=" * 60)
for lookback, signal in momentum_signals.items():
    print(f"  {lookback:2d} days: {signal.name}")

print("\n💡 Strategy Logic:")
print("  • Calculate return over lookback period")
print("  • Standardize to Z-scores (mean=0, std=1)")
print("  • Positive Z-score → Buy (upward trend)")
print("  • Negative Z-score → Sell (downward trend)")

### Calculate Momentum Signals

Let's calculate signals for all instruments at a specific date:

In [ ]:
# Calculate signals as of a specific date
calc_date = date(2024, 11, 1)

signal_values = {}
for lookback, signal in momentum_signals.items():
    values = signal.calculate(instruments, mdp, calc_date)
    signal_values[lookback] = values

# Display as DataFrame
signal_df = pd.DataFrame(signal_values).T
signal_df.index.name = 'Lookback'

print(f"Momentum Signals as of {calc_date}")
print("=" * 80)
print(signal_df.round(3))
print("\nInterpretation:")
print("  • Positive values = upward momentum (buy signal)")
print("  • Negative values = downward momentum (sell signal)")
print("  • Magnitude = strength of trend (Z-score scale)")

# Visualize signal heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(signal_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Signal Strength (Z-score)'},
            linewidths=1, linecolor='black')
plt.title(f'Momentum Signals Across Lookback Periods ({calc_date})', 
          fontsize=14, fontweight='bold')
plt.ylabel('Lookback Period (days)', fontsize=12)
plt.xlabel('Instrument', fontsize=12)
plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print("  • Different lookbacks may give different signals")
print("  • Shorter lookbacks more sensitive to recent moves")
print("  • Longer lookbacks capture persistent trends")

---

## Part 4: Time-Series vs Cross-Sectional Momentum

### Time-Series Momentum (TSMOM)
- Compare each asset to its own past
- Signal = sign(past return) × volatility scaling
- Can be long or short all assets simultaneously

### Cross-Sectional Momentum (XSMOM)
- Rank assets by past returns
- Long top performers, short bottom performers
- Market-neutral by construction

In [ ]:
def calculate_tsmom_signals(raw_signals: Dict[str, float]) -> Dict[str, float]:
    """
    Time-Series Momentum: Sign of past return.
    
    Logic:
    - If past return > 0 → position = +1 (scaled)
    - If past return < 0 → position = -1 (scaled)
    """
    # Raw signals already capture this (mean-centered)
    # Just return the standardized values
    return raw_signals

def calculate_xsmom_signals(raw_signals: Dict[str, float]) -> Dict[str, float]:
    """
    Cross-Sectional Momentum: Rank-based.
    
    Logic:
    - Rank all assets by past return
    - Top half get positive weights
    - Bottom half get negative weights
    - Ensures market neutrality
    """
    # Sort by signal value
    sorted_items = sorted(raw_signals.items(), key=lambda x: x[1])
    n = len(sorted_items)
    
    # Assign ranks (normalized to [-1, 1])
    xsmom = {}
    for i, (inst, _) in enumerate(sorted_items):
        # Rank from -1 (worst) to +1 (best)
        rank = (2 * i / (n - 1)) - 1 if n > 1 else 0
        xsmom[inst] = rank
    
    return xsmom

# Compare both approaches
lookback = 20  # Use 20-day momentum
raw_signals = signal_values[lookback]

tsmom = calculate_tsmom_signals(raw_signals)
xsmom = calculate_xsmom_signals(raw_signals)

# Display comparison
comparison = pd.DataFrame({
    'Raw Return': raw_signals,
    'TSMOM Signal': tsmom,
    'XSMOM Signal': xsmom
})

print("Time-Series vs Cross-Sectional Momentum")
print("=" * 60)
print(comparison.round(3))
print()
print(f"TSMOM Sum: {sum(tsmom.values()):.3f} (can be non-zero)")
print(f"XSMOM Sum: {sum(xsmom.values()):.3f} (market neutral)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TSMOM
ax1 = axes[0]
colors1 = ['green' if v > 0 else 'red' for v in tsmom.values()]
ax1.barh(list(tsmom.keys()), list(tsmom.values()), color=colors1, alpha=0.7)
ax1.axvline(0, color='black', linestyle='--', alpha=0.5)
ax1.set_title('Time-Series Momentum\n(Absolute Trends)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Signal Strength', fontsize=11)
ax1.grid(True, alpha=0.3, axis='x')

# XSMOM
ax2 = axes[1]
colors2 = ['green' if v > 0 else 'red' for v in xsmom.values()]
ax2.barh(list(xsmom.keys()), list(xsmom.values()), color=colors2, alpha=0.7)
ax2.axvline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('Cross-Sectional Momentum\n(Relative Strength)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Signal Strength', fontsize=11)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n💡 Key Differences:")
print("  TSMOM: Captures absolute trends (all assets can be long/short)")
print("  XSMOM: Captures relative strength (always long strongest, short weakest)")
print("  XSMOM: Market-neutral by construction (hedge against systematic risk)")

---

## Part 5: Signal Combination

Combining multiple momentum lookbacks can improve performance by:
1. Diversifying across timeframes
2. Reducing noise from single-period signals
3. Capturing both short and long-term trends

We'll use three combination methods from `SignalCombiner`:
1. **Equal Weight** - Simple average
2. **IC-Weighted** - Weight by historical predictive power
3. **Orthogonalization** - Remove redundancy between signals

In [ ]:
# Prepare signals for combination
signals_dict = {f"mom_{lb}d": signal_values[lb] for lb in lookback_periods}

# Create combiner
combiner = SignalCombiner()

# Method 1: Equal Weight
combined_equal = combiner.combine(signals_dict, method='equal')

# Method 2: IC-Weighted (assuming different IC for each lookback)
# Shorter lookbacks typically have lower IC
ic_estimates = {
    'mom_5d': 0.02,   # Very short-term, low IC
    'mom_10d': 0.03,  # Short-term
    'mom_20d': 0.05,  # Medium-term, highest IC
    'mom_60d': 0.04   # Long-term
}
combined_ic = combiner.combine(signals_dict, method='ic_weighted', ic_estimates=ic_estimates)

# Method 3: Orthogonalization
combined_ortho = combiner.combine(signals_dict, method='orthogonal')

# Compare all methods
combination_df = pd.DataFrame({
    'Equal Weight': combined_equal,
    'IC-Weighted': combined_ic,
    'Orthogonalized': combined_ortho
})

print("Signal Combination Comparison")
print("=" * 70)
print(combination_df.round(3))
print()
print("Method Descriptions:")
print("  • Equal Weight: Simple average of all signals")
print("  • IC-Weighted: Emphasizes 20-day (highest IC)")
print("  • Orthogonalized: Removes correlation between lookbacks")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (method, signals) in enumerate(combination_df.items()):
    ax = axes[idx]
    colors = ['green' if v > 0 else 'red' for v in signals.values]
    ax.barh(signals.index, signals.values, color=colors, alpha=0.7, edgecolor='black')
    ax.axvline(0, color='black', linestyle='--', alpha=0.5)
    ax.set_title(method, fontsize=13, fontweight='bold')
    ax.set_xlabel('Combined Signal', fontsize=11)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("  • Different methods produce different signal magnitudes")
print("  • IC-weighted emphasizes more predictive lookbacks")
print("  • Orthogonalization reduces redundancy between correlated lookbacks")

---

## Part 6: Run Momentum Backtests

Now let's test each lookback period individually:

In [ ]:
# Define backtest period
start_date = date(2024, 6, 1)
end_date = date(2024, 12, 1)
dates = pd.date_range(start_date, end_date, freq='W').tolist()
dates = [d.date() if hasattr(d, 'date') else d for d in dates]

print(f"Backtest Configuration:")
print(f"  Period: {start_date} to {end_date}")
print(f"  Rebalancing: Weekly ({len(dates)} periods)")
print(f"  Universe: {len(instruments)} contracts")
print()
print("Running backtests for each lookback period...\n")

# Run backtest for each momentum signal
momentum_results = {}

for lookback in lookback_periods:
    print(f"Testing {lookback}-day momentum...")
    
    # Reset market data state
    np.random.seed(42)
    mdp_test = MomentumMockMDP(
        base_rate=5.0,
        carry_spread=0.10,
        momentum_strength=0.3,
        seed=42
    )
    
    # Generate initial history
    for inst in instruments:
        for _ in range(200):
            mdp_test.futures_price(inst)
    
    # Create backtest with FuturesAdapter and momentum signal
    backtest = Backtest(
        mdp=mdp_test,
        adapter=FuturesAdapter(mdp_test),
        signals=momentum_signals[lookback],
        risk_aversion=1.0,
        long_only=False,
        min_history=lookback + 5
    )
    
    try:
        result = backtest.run(contracts=instruments, dates=dates)
        momentum_results[lookback] = result
        print(f"  ✓ Complete - Sharpe: {result.sharpe_ratio:.3f}\n")
    except Exception as e:
        print(f"  ✗ Failed: {e}\n")

print("=" * 70)
print("✓ All momentum backtests complete!")

### Compare Momentum Lookback Performance

In [ ]:
# Create performance comparison
perf_comparison = pd.DataFrame({
    'Lookback (days)': list(momentum_results.keys()),
    'Sharpe Ratio': [r.sharpe_ratio for r in momentum_results.values()],
    'IC': [r.ic for r in momentum_results.values()],
    'Total Return': [r.total_return for r in momentum_results.values()],
    'Volatility': [r.returns.std() for r in momentum_results.values()]
}).set_index('Lookback (days)')

print("Momentum Lookback Performance Comparison")
print("=" * 70)
print(perf_comparison.round(4))
print()
print("Best Performers:")
print(f"  Highest Sharpe: {perf_comparison['Sharpe Ratio'].idxmax()}-day")
print(f"  Highest IC: {perf_comparison['IC'].idxmax()}-day")
print(f"  Highest Return: {perf_comparison['Total Return'].idxmax()}-day")

# Visualize performance metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sharpe Ratio
ax1 = axes[0, 0]
ax1.plot(perf_comparison.index, perf_comparison['Sharpe Ratio'], 
         marker='o', linewidth=2.5, markersize=10, color='steelblue')
ax1.axhline(0, color='red', linestyle='--', alpha=0.3)
ax1.axhline(1, color='green', linestyle='--', alpha=0.3, label='Good (>1.0)')
ax1.set_title('Sharpe Ratio by Lookback', fontsize=13, fontweight='bold')
ax1.set_xlabel('Lookback Period (days)', fontsize=11)
ax1.set_ylabel('Sharpe Ratio', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Information Coefficient
ax2 = axes[0, 1]
ax2.plot(perf_comparison.index, perf_comparison['IC'],
         marker='s', linewidth=2.5, markersize=10, color='green')
ax2.axhline(0, color='red', linestyle='--', alpha=0.3)
ax2.axhline(0.05, color='green', linestyle='--', alpha=0.3, label='Good (>0.05)')
ax2.set_title('Information Coefficient by Lookback', fontsize=13, fontweight='bold')
ax2.set_xlabel('Lookback Period (days)', fontsize=11)
ax2.set_ylabel('IC', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.legend()

# 3. Total Return
ax3 = axes[1, 0]
returns_pct = perf_comparison['Total Return'] * 100
colors = ['green' if r > 0 else 'red' for r in returns_pct]
ax3.bar(perf_comparison.index, returns_pct, color=colors, alpha=0.7, edgecolor='black')
ax3.axhline(0, color='black', linestyle='-', alpha=0.3)
ax3.set_title('Total Return by Lookback', fontsize=13, fontweight='bold')
ax3.set_xlabel('Lookback Period (days)', fontsize=11)
ax3.set_ylabel('Total Return (%)', fontsize=11)
ax3.grid(True, alpha=0.3, axis='y')

# 4. Cumulative returns comparison
ax4 = axes[1, 1]
for lookback, result in momentum_results.items():
    cumulative = (1 + result.returns).cumprod()
    ax4.plot(cumulative.index, cumulative.values, 
             label=f"{lookback}d", linewidth=2, alpha=0.7)
ax4.axhline(1, color='black', linestyle='--', alpha=0.3)
ax4.set_title('Cumulative Returns Comparison', fontsize=13, fontweight='bold')
ax4.set_ylabel('Portfolio Value', fontsize=11)
ax4.legend(loc='best', fontsize=9)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Findings:")
print("  • Optimal lookback depends on market regime")
print("  • 20-60 day lookbacks typically perform best (academic consensus)")
print("  • Very short lookbacks (5d) may be too noisy")
print("  • Combining multiple lookbacks often improves robustness")

---

## Part 7: Momentum vs Carry Comparison

How does momentum compare to the traditional carry strategy?

In [ ]:
print("Running carry strategy for comparison...\n")

# Reset market data
np.random.seed(42)
mdp_carry = MomentumMockMDP(
    base_rate=5.0,
    carry_spread=0.10,
    momentum_strength=0.3,
    seed=42
)

# Generate history
for inst in instruments:
    for _ in range(200):
        mdp_carry.futures_price(inst)

# Note: We would normally use CarrySignal here, but since our mock data
# doesn't have proper carry structure (next_price, roll_date), we'll
# simulate a simple carry strategy result for comparison

# Use best momentum result for comparison
best_momentum_lookback = perf_comparison['Sharpe Ratio'].idxmax()
momentum_result = momentum_results[best_momentum_lookback]

# Create comparison (using simulated carry performance)
# In real implementation, you would run:
# carry_backtest = Backtest(mdp=mdp_carry, ...)
# carry_result = carry_backtest.run(...)

print(f"Comparing {best_momentum_lookback}-day Momentum vs Carry Strategy")
print("=" * 70)
print(f"\nMomentum ({best_momentum_lookback}d):")
print(f"  Sharpe Ratio: {momentum_result.sharpe_ratio:.3f}")
print(f"  IC: {momentum_result.ic:.3f}")
print(f"  Total Return: {momentum_result.total_return:.2%}")
print(f"  Volatility: {momentum_result.returns.std():.4f}")

print("\n💡 Expected Characteristics:")
print("\nMomentum:")
print("  • Pros: Works in trending markets, diversifies carry")
print("  • Cons: High turnover, momentum crashes, reversal risk")
print("  • IC: 0.02-0.05 (moderate)")
print("\nCarry:")
print("  • Pros: Low turnover, stable income, works in range-bound markets")
print("  • Cons: Fails in trends, negative skew (crash risk)")
print("  • IC: 0.05-0.10 (good)")
print("\nCombination Strategy:")
print("  • Best of both: carry for income, momentum for trends")
print("  • Lower correlation → diversification benefit")
print("  • Expected IR improvement: 20-40%")

---

## Part 8: Parameter Sensitivity Analysis

How sensitive is momentum performance to the choice of lookback period?

In [ ]:
# Test wider range of lookback periods
extended_lookbacks = [3, 5, 10, 15, 20, 30, 40, 60, 90]

print("Running sensitivity analysis across lookback periods...")
print(f"Testing {len(extended_lookbacks)} different lookbacks\n")

sensitivity_results = {}

for lookback in extended_lookbacks:
    print(f"  Testing {lookback:2d}-day lookback...", end='')
    
    # Reset data
    np.random.seed(42)
    mdp_test = MomentumMockMDP(base_rate=5.0, carry_spread=0.10, 
                                momentum_strength=0.3, seed=42)
    
    for inst in instruments:
        for _ in range(200):
            mdp_test.futures_price(inst)
    
    # Create momentum signal for this lookback
    signal = MomentumSignal(
        name=f"momentum_{lookback}d",
        lookback_days=lookback,
        method='simple',
        annualize=False,
        standardize=True
    )
    
    # Run backtest with FuturesAdapter and momentum signal
    backtest = Backtest(
        mdp=mdp_test,
        adapter=FuturesAdapter(mdp_test),
        signals=signal,
        risk_aversion=1.0,
        long_only=False,
        min_history=lookback + 5
    )
    
    try:
        result = backtest.run(contracts=instruments, dates=dates)
        sensitivity_results[lookback] = {
            'sharpe': result.sharpe_ratio,
            'ic': result.ic,
            'return': result.total_return,
            'volatility': result.returns.std()
        }
        print(f" Sharpe={result.sharpe_ratio:.3f}")
    except Exception as e:
        print(f" Failed: {e}")

print("\n✓ Sensitivity analysis complete!")

---

## Part 9: Turnover Analysis

Momentum strategies trade frequently. Let's analyze turnover and its impact on costs.

**Turnover** = Sum of absolute weight changes between rebalancing periods

In [ ]:
def calculate_turnover(weights_df: pd.DataFrame) -> pd.Series:
    """
    Calculate portfolio turnover.
    
    Turnover = sum of abs(weight[t] - weight[t-1]) for all instruments
    """
    turnover = []
    
    for i in range(1, len(weights_df)):
        weight_change = (weights_df.iloc[i] - weights_df.iloc[i-1]).abs().sum()
        turnover.append(weight_change)
    
    return pd.Series(turnover, index=weights_df.index[1:])

# Calculate turnover for each momentum strategy
turnover_analysis = {}

for lookback, result in momentum_results.items():
    turnover = calculate_turnover(result.weights)
    turnover_analysis[f"{lookback}d"] = {
        'mean': turnover.mean(),
        'std': turnover.std(),
        'max': turnover.max(),
        'total': turnover.sum()
    }

turnover_df = pd.DataFrame(turnover_analysis).T

print("Portfolio Turnover Analysis")
print("=" * 70)
print(turnover_df.round(4))
print()
print("Interpretation:")
print("  • Mean: Average weekly turnover (1.0 = complete portfolio flip)")
print("  • Total: Cumulative turnover over backtest period")
print("  • Higher turnover = higher transaction costs")

# Visualize turnover
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Time series of turnover
ax1 = axes[0]
for lookback, result in momentum_results.items():
    turnover = calculate_turnover(result.weights)
    ax1.plot(turnover.index, turnover.values, 
             label=f"{lookback}d", linewidth=2, alpha=0.7, marker='o', markersize=4)
ax1.set_title('Portfolio Turnover Over Time', fontsize=14, fontweight='bold')
ax1.set_ylabel('Turnover (% of Portfolio)', fontsize=12)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Average turnover by lookback
ax2 = axes[1]
lookbacks = [int(k[:-1]) for k in turnover_df.index]
mean_turnover = turnover_df['mean'].values
bars = ax2.bar(lookbacks, mean_turnover, alpha=0.7, color='coral', edgecolor='black')
ax2.set_title('Average Turnover by Lookback Period', fontsize=14, fontweight='bold')
ax2.set_xlabel('Lookback Period (days)', fontsize=12)
ax2.set_ylabel('Mean Weekly Turnover', fontsize=12)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, mean_turnover):
    ax2.text(bar.get_x() + bar.get_width()/2, val,
             f'{val:.1%}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Turnover Insights:")
print("  • Shorter lookbacks → higher turnover (more sensitive to noise)")
print("  • Longer lookbacks → lower turnover (more stable signals)")
print("  • Typical momentum turnover: 30-70% per rebalance")
print("  • Compare to carry: ~10-20% (much more stable)")

# Estimate transaction cost impact
print("\n" + "="*70)
print("Transaction Cost Impact Estimation")
print("="*70)

cost_per_turn = 0.0002  # 2 bps round-trip (futures are cheap)

print(f"\nAssuming {cost_per_turn*10000:.1f} bps round-trip transaction cost:")
print()

for lookback, result in momentum_results.items():
    turnover = calculate_turnover(result.weights)
    total_cost = turnover.sum() * cost_per_turn
    annual_cost = (total_cost / len(dates)) * 52  # Annualized
    
    gross_return = result.total_return
    net_return = gross_return - total_cost
    
    print(f"{lookback}-day momentum:")
    print(f"  Gross return: {gross_return:>7.2%}")
    print(f"  Transaction costs: {total_cost:>7.2%}")
    print(f"  Net return: {net_return:>7.2%}")
    print(f"  Cost drag (annualized): {annual_cost:>7.2%}")
    print()

print("💡 Key Takeaway:")
print("  • Transaction costs can significantly impact momentum strategies")
print("  • Longer lookbacks → lower costs → potentially better net returns")
print("  • Always evaluate strategies on an after-cost basis")
print("  • Consider rebalancing frequency (weekly vs monthly)")

---

## Part 10: Combined Carry + Momentum Strategy

Let's combine momentum with carry for a diversified multi-signal strategy:

In [ ]:
print("Creating Combined Carry + Momentum Strategy")
print("=" * 70)
print()
print("Strategy Design:")
print("  • Signal 1: Carry (stable income, low turnover)")
print("  • Signal 2: 20-day Momentum (trend capture)")
print("  • Signal 3: 60-day Momentum (long-term trend)")
print("  • Combination: Equal weight (diversification)")
print()
print("Expected Benefits:")
print("  • Carry works in range-bound markets")
print("  • Momentum works in trending markets")
print("  • Low correlation → diversification benefit")
print("  • IR improvement: 20-40% vs single signal")
print()

# Note: In a real implementation with proper market data,
# you would use the multi_signal template:
#
# combined_strategy = quick_strategy(
#     'multi_signal',
#     instruments=instruments
# )
#
# This template combines CarrySignal + MomentumSignal automatically

print("\n💡 Implementation Guidance:")
print()
print("Using the Strategy Factory:")
print()
print("```python")
print("from Strategies.Registry import quick_strategy")
print()
print("# Create multi-signal strategy")
print("strategy = quick_strategy(")
print("    'multi_signal',")
print("    instruments=['SFRZ4', 'SFRH5', 'SFRM5', 'SFRU5']")
print(")")
print()
print("# Run backtest")
print("backtest = Backtest(mdp=mdp, risk_aversion=1.0, long_only=False)")
print("result = backtest.run(contracts=instruments, dates=dates)")
print("```")
print()
print("Or using YAML configuration (see examples/yaml_strategy_example.py):")
print()
print("```yaml")
print("name: carry_momentum_combo")
print("signals:")
print("  - type: CarrySignal")
print("    params:")
print("      name: futures_carry")
print("  - type: MomentumSignal")
print("    params:")
print("      name: momentum_20d")
print("      lookback_days: 20")
print("  - type: MomentumSignal")
print("    params:")
print("      name: momentum_60d")
print("      lookback_days: 60")
print("signal_combination: equal  # or 'ic_weighted', 'orthogonal'")
print("```")

---

## Summary and Key Takeaways

### What We Learned

1. **Momentum Anomaly**
   - Strong academic evidence across asset classes
   - Driven by behavioral biases and market microstructure
   - Works best with 1-12 month lookbacks

2. **Multi-Period Analysis**
   - Different lookbacks capture different time scales
   - 20-40 day lookbacks typically optimal for futures
   - Combining multiple lookbacks improves robustness

3. **Time-Series vs Cross-Sectional**
   - TSMOM: Absolute trends (each asset vs its past)
   - XSMOM: Relative strength (assets vs each other)
   - Both approaches valid, serve different purposes

4. **Signal Combination**
   - Equal weight: Simple and robust
   - IC-weighted: Emphasizes strong signals
   - Orthogonalization: Removes redundancy

5. **Performance Characteristics**
   - Expected IC: 0.02-0.05 (moderate)
   - Expected Sharpe: 0.5-1.0 (after costs)
   - High turnover: 30-70% per rebalance
   - Transaction costs significant (must monitor)

6. **Practical Considerations**
   - Always evaluate after transaction costs
   - Consider rebalancing frequency (weekly vs monthly)
   - Combine with carry for diversification
   - Monitor for momentum crashes (rapid reversals)

### Best Practices

✅ **Do:**
- Use multiple lookback periods
- Combine with other signals (carry, value)
- Monitor turnover and transaction costs
- Test parameter sensitivity
- Use volatility scaling for position sizing

❌ **Don't:**
- Rely on a single lookback period
- Ignore transaction costs
- Over-optimize parameters (curve fitting)
- Trade too frequently (costs compound)
- Assume past momentum = future momentum

### Next Steps

1. **Implement with Real Data**: Connect to actual market data provider
2. **Add Risk Management**: Volatility targeting, position limits
3. **Test Transaction Costs**: Use realistic cost models
4. **Regime Detection**: Identify when momentum works best
5. **Cross-Asset Momentum**: Extend to bonds, commodities, FX

### Further Reading

**Key Papers:**
- Moskowitz, Ooi & Pedersen (2012): "Time Series Momentum"
- Jegadeesh & Titman (1993): "Returns to Buying Winners and Selling Losers"
- Asness, Moskowitz & Pedersen (2013): "Value and Momentum Everywhere"

**Books:**
- Grinold & Kahn (2000): "Active Portfolio Management" - Chapter on momentum
- Carver (2015): "Systematic Trading" - Practical momentum implementation

### Related Notebooks

- `01_getting_started.ipynb` - Framework introduction
- `02_strategy_comparison.ipynb` - Compare multiple strategies
- `03_parameter_tuning.ipynb` - Systematic optimization
- `05_cross_asset_integration.ipynb` - Multi-asset strategies

---

## Congratulations!

You now have a comprehensive understanding of momentum strategies in futures markets. You've learned:

✅ The behavioral and empirical foundations of momentum

✅ How to implement multi-period momentum signals

✅ Time-series vs cross-sectional approaches

✅ Signal combination techniques

✅ Performance analysis and parameter sensitivity

✅ Turnover and transaction cost impact

Happy trading! 🚀